# Agent 4 — Wearable Agent (Package Edition)

Post-extraction demo for Agent 4. All class and function definitions now live in the `immunosense.agents.wearable` package.

**Architecture (unchanged):**
- L1: Mock data generator (HealthKit/Health Connect/Fitbit/Oura integration deferred)
- L2: Hampel artifact removal, Akima interpolation, ENMO sleep/wake
- L3: 29-feature engineering — HRV / sleep / temp / cardio / activity / respiratory / TADI / composite / quality
- L4: Personal baselines via shared `RobustBaselineTracker`
- L5: Single-metric + named composite alerts (autoimmune_prodrome, acute_stress_response)
- L6: 29-dim output vector + wearable_stress_score + physiological state labels
- L7: BaseAgent contract

**Validations performed:**
1. Single-night smoke test (normal vs prodrome)
2. 30-night trajectory with embedded flares (acute stress at night 10, prodrome nights 20-21)
3. Alert evaluation across the trajectory


In [ ]:
# === Package imports ===
import numpy as np
import pandas as pd

from immunosense.agents.wearable import (
    WearableAgent,
    MockWearableGenerator,
    FEATURE_NAMES,
    HampelFilter,
    akima_interpolate,
    enmo_sleep_wake,
    compute_hrv_features,
    compute_sleep_features,
    compute_tadi,
)

print('Imports OK')
print(f'WearableAgent.agent_id     = {WearableAgent.agent_id}')
print(f'WearableAgent.output_dim   = {WearableAgent.output_dim}')
print(f'WearableAgent.agent_version = {WearableAgent.agent_version}')
print(f'Feature vector dim: {len(FEATURE_NAMES)}')


In [ ]:
# === Section 1: Single-night smoke test (normal vs prodrome) ===
gen = MockWearableGenerator(patient_id='p001', seed=42)
night_normal, rr_normal = gen.generate_night(0, flare_state='normal')
night_prodrome, rr_prodrome = gen.generate_night(1, flare_state='prodrome')

print('Normal night:')
print(f'  Mean HR:        {night_normal["hr"].mean():.1f} bpm')
print(f'  Mean skin temp: {night_normal["skin_temp"].mean():.2f} C')
print(f'  Stage dist:     {night_normal["sleep_stage"].value_counts().to_dict()}')
print(f'  RR intervals:   {len(rr_normal)} beats')
print()
print('Prodrome night:')
print(f'  Mean HR:        {night_prodrome["hr"].mean():.1f} bpm (expect ~4 higher)')
print(f'  Mean skin temp: {night_prodrome["skin_temp"].mean():.2f} C (expect ~0.4 higher)')
print(f'  RR intervals:   {len(rr_prodrome)} beats (HRV reduced)')


In [ ]:
# === Section 2: Preprocessing utilities ===
# Hampel filter on RR intervals
hampel = HampelFilter(window_size=11, k_threshold=3.0)
rr_arr = np.array(rr_prodrome)
rr_clean, rr_outlier_mask = hampel.filter(rr_arr)
print(f'Hampel filter: {rr_outlier_mask.sum()} of {len(rr_arr)} RR intervals '
      f'flagged as artifacts ({rr_outlier_mask.mean()*100:.2f}%)')

# ENMO sleep/wake on the prodrome night
labels = enmo_sleep_wake(night_prodrome['enmo'].values, night_prodrome['hr'].values)
print(f'ENMO sleep/wake: {(labels == "sleep").sum()} sleep mins, '
      f'{(labels == "wake").sum()} wake mins')

# Akima interpolation on a synthetic signal with a gap
ts = np.arange(20).astype(float)
vals = np.array([60, 61, 62, 63, np.nan, np.nan, np.nan, 80, 81, 82,
                  83, 82, 81, 80, 79, 78, 77, 76, 75, 74])
filled = akima_interpolate(ts, vals, ts)
print(f'Akima: filled {np.isnan(vals).sum()} missing points, '
      f'min={filled.min():.1f}, max={filled.max():.1f}  (no overshoot)')


In [ ]:
# === Section 3: 30-night integration test ===
#
# Reproduces the original notebook's end-to-end test:
#   - 30 nights of nightly readings
#   - acute_stress night at index 10
#   - prodrome nights at indices 20-21
#
# Expected:
#   - autoimmune_prodrome composite fires on or near night 20-21
#   - acute_stress_response composite fires on night 10
#   - At least 1 critical alert total
#
# WARNING: this takes ~3 minutes (HRV computation over ~30k RR intervals/night).
agent = WearableAgent()
gen = MockWearableGenerator(patient_id='p001', seed=42)

flare_schedule = ['normal'] * 30
flare_schedule[10] = 'acute_stress'
flare_schedule[20] = 'prodrome'
flare_schedule[21] = 'prodrome'

outputs = []
for night_idx, state in enumerate(flare_schedule):
    night_df, rr_intervals = gen.generate_night(night_idx, flare_state=state)
    out = agent.process({
        'night_df': night_df,
        'rr_intervals': rr_intervals,
        'night_idx': night_idx,
        'is_flare': state != 'normal',
    })
    outputs.append((state, out))

# Summary
n_critical = sum(1 for s, o in outputs if any(a['severity'] == 'critical' for a in o.alerts))
n_warning = sum(1 for s, o in outputs if any(a['severity'] == 'warning' for a in o.alerts))
n_prodrome = sum(1 for s, o in outputs if any(a['name'] == 'autoimmune_prodrome' for a in o.alerts))
n_acute = sum(1 for s, o in outputs if any(a['name'] == 'acute_stress_response' for a in o.alerts))

print(f'Total nights:                  {len(outputs)}')
print(f'Nights with critical alerts:   {n_critical}')
print(f'Nights with warning alerts:    {n_warning}')
print(f'autoimmune_prodrome firings:   {n_prodrome}')
print(f'acute_stress_response firings: {n_acute}')


In [ ]:
# === Section 4: Inspect a prodrome night's full output ===
prodrome_state, prodrome_out = next((s, o) for s, o in outputs if s == 'prodrome')
print(f'Example prodrome night AgentOutput:')
print(f'  agent_id:     {prodrome_out.agent_id}')
print(f'  vector_dim:   {prodrome_out.vector_dim}')
print(f'  confidence:   {prodrome_out.confidence:.3f}')
print(f'  trace_id:     {prodrome_out.trace_id}')
print(f'  alerts:       {len(prodrome_out.alerts)}')
for a in prodrome_out.alerts:
    print(f'    - {a["name"]:<30} [{a["severity"]}]')
print(f'  states:       {prodrome_out.data["physiological_states"]}')
print(f'  stress_score: {prodrome_out.data["reading"]["wearable_stress_score"]:.3f}')
print()
print(f'Agent health: {agent.get_status()}')
